In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [32]:
class Value:
    def __init__(self, data, _children = (), _op = ''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda : None

    def __repr__(self):
        return f"Value(data: {self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return (-1.0 * self)
        
    def __sub__(self, other):
        return self + (-other)

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self, ), '**')
        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = (math.exp(2*self.data) - 1)/(math.exp(2*self.data) + 1)
        out = Value(t, (self, ), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            visited.add(v)
            for child in v._prev:
                if child not in visited:
                    build_topo(child)
            topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [65]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for l in self.layers:
            params.extend(l.parameters())
        return params

In [94]:
n = MLP(3, [4, 4, 1])

In [95]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]

In [97]:
for k in range(40):
    for p in n.parameters():
        p.grad = 0.0
    
    ypred = [n(x) for x in xs]
    loss = sum((yout - Value(ygt))**2 for yout, ygt in zip(ypred, ys))

    loss.backward()

    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)

0 0.024278160154218357
1 0.02328607512999471
2 0.022370464860423305
3 0.021523024647966318
4 0.020736586090999272
5 0.020004932331854876
6 0.01932264773923548
7 0.018684994844422268
8 0.01808781299706174
9 0.017527434443022648
10 0.017000614463374576
11 0.016504472928266626
12 0.016036445168892185
13 0.015594240495982055
14 0.01517580702458322
15 0.014779301724627287
16 0.014403064821632623
17 0.014045597834351407
18 0.013705544665721043
19 0.013381675267317537
20 0.013072871481151854
21 0.012778114730342679
22 0.01249647528523386
23 0.012227102876461794
24 0.011969218463326258
25 0.01172210699615162
26 0.011485111036394345
27 0.011257625119049939
28 0.0110390907592203
29 0.010828992019168125
30 0.010626851564301537
31 0.010432227146725216
32 0.010244708463589502
33 0.010063914344739244
34 0.009889490230334303
35 0.009721105904361059
36 0.00955845345443226
37 0.009401245432101323
38 0.00924921319120121
39 0.009102105384540513
